In [1]:
!pip install grpcio grpcio-tools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 13.5 MB/s  0:00:00 14.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 9.3 MB/s  0:00:01a 0:00:010:00:01:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.35.0
    Uninstalling protobuf-7.35.0:
      Successfully uninstalled protobuf-7.35.0
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.80.0
    Uninstalling grpcio-1.80.0:
      Successfully uninstalled grpcio-1.80.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [grpcio-tools]0m 1/3 [grpcio]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cirq-google 1.6.1 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
weaviate-client 4.19.0 requires grpcio<1.80.0,>=1.59.5, but you have grpcio 1.82.1 which is incompatible.
weaviate-client 4.19.0 requi

In [2]:
!python -m grpc_tools.protoc -I. --python_out=. --pyi_out=. --grpc_python_out=. grpc_test.proto

In [ ]:
# gRPC server

from concurrent import futures
import grpc
import grpc_test_pb2
import grpc_test_pb2_grpc

# Mock database
PRODUCTS_DB = {
    "A100": {"name": "Mechanical Keyboard", "price": 89.99, "stock": 14},
    "B200": {"name": "Ergonomic Wireless Mouse", "price": 49.50, "stock": 42},
    "C300": {"name": "UltraWide 34-inch Monitor", "price": 349.99, "stock": 8}
}

class ProductServicer(grpc_test_pb2_grpc.ProductServiceServicer):
    
    def GetProductDetails(self, request, context):
        print(f"[Server] Received request for Product ID: {request.product_id}")
        
        product_id = request.product_id
        
        if product_id in PRODUCTS_DB:
            data = PRODUCTS_DB[product_id]
            # Return the generated protobuf response object
            return grpc_test_pb2.ProductResponse(
                product_id=product_id,
                name=data["name"],
                price=data["price"],
                stock_quantity=data["stock"]
            )
        else:
            # If not found, return a gRPC error status
            context.set_code(grpc.StatusCode.NOT_FOUND)
            context.set_details(f"Product {product_id} not found.")
            return grpc_test_pb2.ProductResponse()

def serve():
    # Create the gRPC server with a thread pool to handle concurrent requests
    server = grpc.server(futures.ThreadPoolExecutor(max_workers=10))
    grpc_test_pb2_grpc.add_ProductServiceServicer_to_server(ProductServicer(), server)
    
    # Listen on port 50051
    server.add_insecure_port("[::]:50051")
    print("gRPC Server is running on port 50051...")
    server.start()
    server.wait_for_termination()

if __name__ == "__main__":
    serve()

gRPC Server is running on port 50051...


In [4]:
# gRPC client

import grpc
import grpc_test_pb2
import grpc_test_pb2_grpc

def run():
    # Open a gRPC channel to the server
    with grpc.insecure_channel("localhost:50051") as channel:
        # Create a stub (client)
        stub = grpc_test_pb2_grpc.ProductServiceStub(channel)
        
        # 1. Query a valid product
        print("\n[Client] Fetching details for Product A100...")
        try:
            request = grpc_test_pb2.ProductRequest(product_id="A100")
            response = stub.GetProductDetails(request)
            print(f"Success! Name: {response.name} | Price: ${response.price} | Stock: {response.stock_quantity}")
        except grpc.RpcError as e:
            print(f"Error: {e.details()}")

        # 2. Query a non-existent product to trigger the error path
        print("\n[Client] Fetching details for non-existent Product XYZ...")
        try:
            request = grpc_test_pb2.ProductRequest(product_id="XYZ")
            response = stub.GetProductDetails(request)
        except grpc.RpcError as e:
            # We catch the custom error code sent by our server
            print(f"Failed expectedly: Code = {e.code()} | Details = {e.details()}")

if __name__ == "__main__":
    run()


[Client] Fetching details for Product A100...
Success! Name: Mechanical Keyboard | Price: $89.99 | Stock: 14

[Client] Fetching details for non-existent Product XYZ...
Failed expectedly: Code = StatusCode.NOT_FOUND | Details = Product XYZ not found.
